# 15 Remote work and moving data

<div class="bp-banner">
  <div class="bp-series">Introduction to the Bash Shell</div>
  <div style="display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;">
    <span class="bp-title">Part IV — Working on a cluster</span>
    <span class="bp-meta">Notebook&nbsp;15</span>
  </div>
  <div style="margin-top:10px;max-width:62ch;color:#46506b;">
    Operating a machine you are not sitting at: reaching it with <code>ssh</code>,
    carrying data across with <code>tar</code>/<code>rsync</code>, and keeping long
    work alive and monitored after you disconnect.
  </div>
  <div class="bp-rule" style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
    <span class="bp-meta">Raymond Amador</span>
    <span class="bp-meta">v0.1.0&nbsp;·&nbsp;CC&nbsp;BY&nbsp;4.0 (text) / MIT (code)</span>
  </div>
</div>

In [1]:
# Hidden setup: stand at the repo root, source the validation gate. data/ is
# read-only; everything we archive, sync, and link lives in a fresh scratch/.
ROOT="$PWD"; while [ ! -f "$ROOT/tools/check.sh" ] && [ "$ROOT" != "/" ]; do ROOT="$(dirname "$ROOT")"; done
source "$ROOT/tools/check.sh"
set +H
# Turn off monitor mode so backgrounded jobs don't print async "[N] PID" /
# "Terminated" notices that would arrive after a cell and confuse the kernel.
set +m
cd "$ROOT"

## What this notebook is about

Everything so far you could do on the machine in front of you. The cluster changes
one thing: **you are not sitting at it.** It is a computer somewhere else — in a
machine room, reached over the network — and that single fact creates three needs.
You have to **get on and stay on** it (`ssh`, `tmux`). You have to **move your data
across** to it and your results back (`tar`, `scp`, `rsync`). And because the real
work takes hours or days, you have to **launch it, disconnect, and come back** to
find it still running and watchable (`&`, `nohup`, `ps`, `top`, `kill`). One frame
— *a machine you're not at* — holds this whole wide toolkit together.

```{admonition} Which cells run here, and which are "type it yourself"
:class: note
There is no second machine inside this page, so some of these tools cannot run in
the grey cells. We label every command:

- A **live grey cell** with output below it runs for real, right here — the
  transfer and process mechanics (`tar`, `rsync` *local-to-local*, `du`, `ps`,
  `ssh-keygen`, …) are all genuine.
- A plain code block tagged **`[your terminal]`** needs a real remote machine or an
  interactive screen (`ssh` to a cluster, `scp`, `tmux`, `top`). Type those in the
  **Practice here** terminal, or — for the connect-and-copy ones — against a real
  cluster like ETH's **Euler**.

The good news: the *mechanics* of moving data are identical whether the
destination is a folder next door or a supercomputer. `rsync` run locally teaches
the real thing; you just add `host:` to the path when the destination is remote.
```

## A. Reaching the machine — `ssh`

**`ssh`** ("secure shell") is the front door to every cluster. Point it at a
machine and it opens a shell there, encrypted end to end, exactly as if you had sat
down at its keyboard:

```{command-card} ssh
```

It comes in two shapes — an interactive session, or a single remote command:

```bash
# [your terminal]
ssh you@euler.ethz.ch        # opens an interactive shell ON Euler
ssh euler squeue             # runs ONE command there and returns its output
```

### Keys, not passwords

Typing your password on every connection is both tedious and less secure than the
standard alternative: a **key pair**. You generate two matching files — a *private*
key you keep, and a *public* key you install on the cluster — and ssh proves your
identity with them automatically. Generating the pair is real, so run it here:

In [2]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [3]:
ssh-keygen -t ed25519 -f scratch/demo_key -N "" -C "you@laptop"

Generating public/private ed25519 key pair.


Your identification has been saved in scratch/demo_key


Your public key has been saved in scratch/demo_key.pub


The key fingerprint is:


SHA256:0D+vdIBxePPzdRUW0F8KnrtaIqNvA4DaSj9sV1ApJ6U you@laptop


...

|...   ..   +.o ..|


|..o   ..o o +..  |


|.  = . .o+ =.    |


|  . o .o..o.     |


+----[SHA256]-----+


That made two files. Look at their permissions — and notice ssh-keygen has already
done the Notebook-11 chmod for you:

In [4]:
ls -l scratch/demo_key scratch/demo_key.pub

-rw------- 1 runner runner 399 Jun 18 19:53 scratch/demo_key


-rw-r--r-- 1 runner runner  92 Jun 18 19:53 scratch/demo_key.pub


The private key is `-rw-------` (mode **600**) — readable only by you. That is not
optional:

:::{admonition} ⚠ ssh refuses a private key others can read
:class: warning
If your private key's file is readable by group or others, **ssh ignores it** and
falls back to asking for a password — the classic "but I set up my key!" confusion.
The fix is the Notebook-11 one: `chmod 600 ~/.ssh/id_ed25519`. Keys are private;
the filesystem has to agree.
:::

You install the *public* half on the cluster with `ssh-copy-id you@euler`, and then
a short **`~/.ssh/config`** turns a long connection string into a one-word alias:

```bash
# [your terminal]
# ~/.ssh/config
Host euler
    HostName euler.ethz.ch
    User your_username
    IdentityFile ~/.ssh/id_ed25519

# now this is all you type:
ssh euler
```

## B. Moving data — `tar`, then `rsync`

Before you transfer a result directory, you usually **bundle and compress** it: one
file moves faster and more reliably than ten thousand. That is **`tar`** (+ gzip):

```{command-card} tar
```

```{command-card} gzip
```

Create an archive, list it without unpacking, and extract it again — all real:

In [5]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/run/logs
printf 'mean = -143.4444\n' > scratch/run/result.txt
printf 'step 1 ok\nstep 2 ok\n' > scratch/run/logs/run.log

In [6]:
tar -czf scratch/run.tar.gz -C scratch run

In [7]:
tar -tzf scratch/run.tar.gz

run/


run/result.txt


run/logs/


run/logs/run.log


In [8]:
mkdir -p scratch/restored && tar -xzf scratch/run.tar.gz -C scratch/restored && find scratch/restored -type f

scratch/restored/run/result.txt


scratch/restored/run/logs/run.log


The archive round-tripped: bundled with `c`, inspected with `t`, unpacked with `x`.
(`gzip` on its own compresses a *single* file — `gzip big.log` → `big.log.gz`; `tar`
is what bundles many.)

### `scp` — the one-shot copy

The simplest remote transfer is **`scp`**, which copies over ssh just like `cp`
copies locally:

```{command-card} scp
```

```bash
# [your terminal]
scp run.tar.gz euler:/cluster/scratch/you/   # push a file up
scp euler:results/summary.txt .              # pull a file down
scp -r mydir euler:~/                         # a whole directory
```

### `rsync` — the workhorse

For anything you will transfer more than once — a results directory you re-sync as a
job produces more — **`rsync`** is the right tool. It copies only what has
*changed*, so the second sync is near-instant. Crucially, its local-to-local form is
the *same tool* you use for the cluster, so we practise it for real:

```{command-card} rsync
```

In [9]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/src; printf 'a\n' > scratch/src/a.txt; printf 'b\n' > scratch/src/b.txt; mkdir -p scratch/dst

In [10]:
rsync -av scratch/src/ scratch/dst/

sending incremental file list


a.txt


b.txt


sent 184 bytes  received 54 bytes  476.00 bytes/sec


total size is 4  speedup is 0.02


Now change one file and re-sync — rsync transfers **only the changed file**, not the
whole set:

In [11]:
printf 'a CHANGED\n' > scratch/src/a.txt; rsync -av scratch/src/ scratch/dst/

sending incremental file list


a.txt


sent 148 bytes  received 35 bytes  366.00 bytes/sec


total size is 12  speedup is 0.07


Only `a.txt` moved the second time. And **preview before you sync** with
`--dry-run`, which lists what *would* happen and touches nothing:

In [12]:
printf 'c\n' > scratch/src/c.txt; rsync -av --dry-run scratch/src/ scratch/dst/

sending incremental file list


c.txt


sent 115 bytes  received 19 bytes  268.00 bytes/sec


total size is 14  speedup is 0.10 (DRY RUN)


In [13]:
ls scratch/dst

a.txt  b.txt


The dry run named `c.txt` as pending, but `scratch/dst` still has only `a` and `b` —
nothing changed. One footgun deserves a hard stop:

:::{admonition} ⚠ The trailing slash, and `--delete`
:class: warning
**Trailing slash:** `rsync -a src/ dst` copies the *contents* of `src` into `dst`;
`rsync -a src dst` copies the *directory* `src` into `dst` (giving `dst/src/…`).
Mixing these up is the most common rsync surprise — be deliberate about the `/`.

**`--delete`:** this makes the destination an exact mirror by **deleting files there
that are gone from the source**. It is exactly what you want for a true mirror — and
a disaster if you point it at the wrong directory. So: `--dry-run` first, *every
time*, and read the list before you run it for real.
:::

## C. Staying on — `tmux` and job control

Here is the problem the cluster forces on you. You start a six-hour job over `ssh`,
close your laptop to go home, and — your connection drops, the shell receives a
**hang-up signal (`SIGHUP`)**, and your job dies with it. Two tools solve this.

### `tmux` — a session that outlives the connection

**`tmux`** runs a terminal *on the remote machine itself* that keeps going when you
detach. You start work inside it, press `Ctrl-b d` to **detach** (the work keeps
running), close your laptop, and later `ssh` back in and `tmux attach` to find
everything exactly as you left it.

```{command-card} tmux
```

```bash
# [your terminal]
ssh euler
tmux                         # start a persistent session
./long_analysis.sh           # launch your work inside it
#   ... press Ctrl-b then d to DETACH; now it's safe to disconnect ...
exit                         # log out of ssh; the job keeps running on Euler

#   ... hours later, from anywhere ...
ssh euler
tmux attach                  # back exactly where you left off
```

### Job control — `&`, `jobs`, `nohup`

For a single command, lighter tools suffice. End a command with **`&`** to run it in
the **background**, freeing your prompt; **`jobs`** lists what is backgrounded. Run
it for real — start a long `sleep`, see it listed, and we will stop it in §D:

In [14]:
cd "$ROOT"

In [15]:
sleep 300 &
echo "backgrounded as PID $!"

[1] 4469


backgrounded as PID 4469


In [16]:
jobs

[1]+  Running                 sleep 300 &


<div class="bp-card">
  <span class="bp-card-cmd">Job control</span> — <span class="bp-card-job">manage a command's foreground/background state (shell syntax, not commands).</span>
  <table>
    <tr><td>cmd &</td><td>run <code>cmd</code> in the background; the prompt returns at once</td></tr>
    <tr><td>jobs</td><td>list the shell's background jobs, with their numbers</td></tr>
    <tr><td>Ctrl-z</td><td>SUSPEND the foreground command (pauses it, hands back the prompt)</td></tr>
    <tr><td>bg  /  fg</td><td>resume a suspended job in the background / foreground</td></tr>
    <tr><td>$!  /  %1</td><td>the PID of the last background command / a job by its number</td></tr>
  </table>
</div>

And **`nohup`** makes a command deaf to that hang-up signal, so it survives logout
even without tmux:

```{command-card} nohup
```

In [17]:
cd "$ROOT/scratch"

In [18]:
nohup sleep 300 > nohup.out 2>&1 &
echo "nohup PID $! — still running after logout; output in nohup.out"

[2] 4470


nohup PID 4470 — still running after logout; output in nohup.out


In [19]:
cd "$ROOT"

(`nohup` parks any output the command would have printed into a file called
`nohup.out`. For real work you usually redirect it somewhere meaningful instead.)

## D. Watching and resources

Now to *watch* those processes and stop them. **`ps`** takes a snapshot of what is
running — pair it with `grep` to find one. Both `sleep`s we backgrounded are alive:

```{command-card} ps
```

In [20]:
ps aux | grep "[s]leep 300"

runner      4469  0.0  0.0   6124  1972 pts/0    S+   19:53   0:00 sleep 300


runner      4470  0.0  0.0   6124  1972 pts/0    S+   19:53   0:00 sleep 300


For a **live**, continuously updating view there is **`top`** (and the friendlier
`htop`) — interactive, so it belongs in a real terminal:

```{command-card} top
```

```bash
# [your terminal]
top        # live process table; press 'q' to quit
htop       # the nicer colour version, if installed
```

To stop a process, **`kill`** sends it a signal by PID. Plain `kill` asks politely
(it can clean up first); `kill -9` is the no-mercy hammer. Let us stop the
background `sleep`s we started — politely:

```{command-card} kill
```

In [21]:
kill %1 %2 2>/dev/null; wait 2>/dev/null; jobs

Both jobs are gone. (When you only have a PID, `kill <PID>`; reach for `kill -9`
only when a process ignores a polite `kill`.)

### `watch` — re-run something to see it change

**`watch`** re-runs a command every couple of seconds in place — perfect for
keeping an eye on a job queue. It is interactive, so it too is a terminal tool:

```{command-card} watch
```

```bash
# [your terminal]
watch squeue          # refresh the SLURM queue every 2s (Notebook 16)
watch -n 5 'ls -l results/ | tail'   # watch results appear, every 5s
```

### Disk awareness — `du` and `df`

Clusters give you a **quota**, and jobs that fill it fail in confusing ways. Two
commands keep you ahead of it. **`du`** measures what your directories use:

```{command-card} du
```

In [22]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/big scratch/small
head -c 200000 /dev/zero > scratch/big/blob.bin
printf 'tiny\n' > scratch/small/note.txt

In [23]:
du -sh scratch/big scratch/small

200K	scratch/big


8.0K	scratch/small


…and **`df`** measures the filesystem (or quota) as a whole:

```{command-card} df
```

In [24]:
df -h "$ROOT" | tail -n 1

/dev/root        72G   56G   16G  78% /


(`du` finds *which directory* is heavy; `df` tells you how full the *disk* is.)

## E. Symlinks — `ln -s`

One last cluster staple. Your big working space (often `/scratch/$USER`) is somewhere
inconvenient, so you make a **symbolic link** — a lightweight signpost — to reach it
by a short name from where you work:

```{command-card} ln
```

In [25]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/realdir; printf 'the real file\n' > scratch/realdir/data.txt

In [26]:
ln -s realdir scratch/link-to-realdir; ls -l scratch/link-to-realdir

lrwxrwxrwx 1 runner runner 7 Jun 18 19:53 scratch/link-to-realdir -> realdir


The `->` in `ls -l` (Notebooks 2–3) shows it is a pointer, not a copy. Follow it and
you reach the real thing:

In [27]:
cat scratch/link-to-realdir/data.txt

the real file


## Exercises

The runnable ones work in a fresh `scratch/`; `data/` stays read-only, and every
backgrounded process is cleaned up. The remote ones are yours to do in a terminal
(the **Practice here** box, or against a real cluster).

### Warm-up 1 (worked) — Archive round-trip

`tar -czf` a directory, list it with `-tzf`, extract it elsewhere with `-xzf`, and
confirm the contents survived.

In [28]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/results
printf 'mean -143.4444\n' > scratch/results/summary.txt
printf 'x\ny\n' > scratch/results/data.csv

In [29]:
# (solution hidden on the public site)


--- contents ---


results/


results/data.csv


results/summary.txt


--- extracted ---


scratch/out/results/data.csv


scratch/out/results/summary.txt


In [30]:
check '[ -f scratch/results.tar.gz ] && [ -f scratch/out/results/summary.txt ] && [ "$(cat scratch/out/results/summary.txt)" = "mean -143.4444" ]' \
      "the archive was created, listed, and extracted with its contents intact"

✓ the archive was created, listed, and extracted with its contents intact


### Warm-up 2 (your turn) — Local `rsync`

`rsync -av` a source directory into a destination; change one file and re-sync (note
only it moves); then `--dry-run` a new file and confirm the destination did **not**
change.

In [31]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/src scratch/dst
printf '1\n' > scratch/src/one.txt; printf '2\n' > scratch/src/two.txt

In [32]:
# (solution hidden on the public site)


sending incremental file list


one.txt


sent 148 bytes  received 35 bytes  366.00 bytes/sec


total size is 8  speedup is 0.04


sending incremental file list


three.txt


sent 123 bytes  received 19 bytes  284.00 bytes/sec


total size is 10  speedup is 0.07 (DRY RUN)


dst still holds:


one.txt  two.txt


In [33]:
check '[ "$(cat scratch/dst/one.txt)" = "1 new" ] && [ -f scratch/dst/two.txt ] && [ ! -f scratch/dst/three.txt ]' \
      "the change synced, and the dry-run previewed three.txt without copying it"

✓ the change synced, and the dry-run previewed three.txt without copying it


### Applied 1 (your turn) — Disk awareness

Make two scratch directories of different sizes, compare them with `du -sh`, and
identify the larger.

In [34]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/heavy scratch/light
head -c 500000 /dev/zero > scratch/heavy/blob.bin
printf 'just a note\n' > scratch/light/note.txt

In [35]:
# (solution hidden on the public site)


496K	scratch/heavy


8.0K	scratch/light


/dev/root        72G   56G   16G  78% /


largest: scratch/heavy


In [36]:
big=$(du -s "$ROOT/scratch/heavy" "$ROOT/scratch/light" | sort -rn | head -n1 | cut -f2)
check 'echo "$big" | grep -q "heavy"' \
      "du sized both directories and identified scratch/heavy as the larger"

✓ du sized both directories and identified scratch/heavy as the larger


### Applied 2 (your turn) — Symlinks

Create a symbolic link to a directory, read the `->` in `ls -l`, and follow the link
to reach a file inside the target.

In [37]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/store; printf 'payload\n' > scratch/store/file.txt

In [38]:
# (solution hidden on the public site)


lrwxrwxrwx 1 runner runner 5 Jun 18 19:53 scratch/shortcut -> store


payload


In [39]:
check '[ -L scratch/shortcut ] && [ "$(readlink scratch/shortcut)" = "store" ] && [ "$(cat scratch/shortcut/file.txt)" = "payload" ]' \
      "the symlink points at store and resolves through to the real file"

✓ the symlink points at store and resolves through to the real file


### Applied 3 (worked) — Background and manage

Background a long command with `&`, find it with `jobs`/`ps`, then `kill` it and
confirm it is gone.

In [40]:
cd "$ROOT"

In [41]:
# (solution hidden on the public site)


[1] 4536


running in background as PID 4536


[1]+  Running                 sleep 300 &


    PID COMMAND


   4536 sleep


after kill, still alive? no


In [42]:
sleep 300 & p=$!
ran=$(ps -p "$p" >/dev/null 2>&1 && echo yes || echo no)
kill "$p"; wait "$p" 2>/dev/null
gone=$(ps -p "$p" >/dev/null 2>&1 && echo no || echo yes)
check '[ "$ran" = "yes" ] && [ "$gone" = "yes" ]' \
      "the command backgrounded, appeared in the process table, then was killed"

[1] 4540


✓ the command backgrounded, appeared in the process table, then was killed


### Remote (terminal / conceptual) — ssh and transfer

No grade here — there is no remote machine on this page. In the **Practice here**
terminal, generate a key with `ssh-keygen` and read the connect-and-copy syntax;
then, if you have access to a real cluster (ETH's **Euler**, say), do the real
thing:

```bash
# [your terminal]
ssh-keygen -t ed25519              # make a key pair
ssh-copy-id you@euler.ethz.ch      # install the public half
ssh euler                          # connect (alias from ~/.ssh/config)
scp run.tar.gz euler:scratch/      # copy a bundle up
rsync -avz results/ euler:results/ # or sync a directory (add -z over the network)
```

### Composite — putting it together (package and sync off the cluster)

The realistic "get my results off the machine" workflow, composing `tar`, `rsync`,
and `du`. Bundle a results directory, sync the bundle to a destination (a local
directory standing in for the remote), and verify it arrived intact.

In [43]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch/job/output scratch/backup
printf 'final energy: -143.4444 eV\n' > scratch/job/output/result.txt
printf 'converged in 6 steps\n' > scratch/job/output/run.log

In [44]:
# (solution hidden on the public site)


sending incremental file list


job-results.tar.gz


sent 343 bytes  received 35 bytes  756.00 bytes/sec


total size is 222  speedup is 0.59


4.0K	scratch/job-results.tar.gz


--- arrived, contents: ---


output/


output/result.txt


output/run.log


In [45]:
check '[ -f scratch/backup/job-results.tar.gz ] && tar -tzf scratch/backup/job-results.tar.gz | grep -q "output/result.txt"' \
      "the results were bundled, synced to the destination, and verified intact"

✓ the results were bundled, synced to the destination, and verified intact


### Optional stretch (terminal) — `tmux`, or a careful `--delete`

No grade. In the **Practice here** terminal: start `tmux`, run something, press
`Ctrl-b d` to detach, then `tmux attach` to return. Or practise a **mirror** sync —
`rsync -av --delete --dry-run src/ dst/` — and read the would-delete list carefully
before ever dropping the `--dry-run` (heed the §B admonition).

## Outlook

You can now reach a cluster, carry your scripts and data on and off it, and keep
long work alive and watched after you disconnect — the whole "machine you're not
at" toolkit. One thing is still missing: on a real cluster you do not just `ssh` in
and run your job by hand, because hundreds of people share the machine. You hand it
to a **scheduler**, which queues it and runs it when resources are free. Next
(Notebook 16): writing a **SLURM submission script** — loading the environment from
Notebook 14 inside it — submitting it with `sbatch`, and watching the queue.

```{compendium-new}
```

<div class="bp-banner" style="margin-top:30px;">
  <div class="bp-series">Take this notebook with you</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Open a <b>live terminal</b> from the &ldquo;Practice here&rdquo; box in any
    section to run the archive, sync, and process commands yourself — and to
    practise the terminal-only ones (<code>tmux</code>, <code>top</code>,
    <code>watch</code>). The published notebooks ship <b>without worked
    solutions</b>; if you would like the reference solutions — to teach from or to
    check your own work — get in touch:
    <a href="mailto:hello@ramador.me">hello@ramador.me</a>.
  </div>
</div>